# T2-18 LangChain RAG 기본 틀 — 통계표 안내

실습 자료(ch2) 흐름을 KOSIS 통계표에 맞춘 골격. 로직은 전부 `src/kosis_rag/`에 있고, 이 노트북은 불러다 쓰기만 한다.

| 실습 자료 | 여기 |
|---|---|
| PyPDFLoader | 실전 1 `data/tables.csv` → `kosis_ai.schemas.TableInfo` 검증 (`kosis_rag.loader.load_catalog`) |
| 청킹 | 없음 — 표 1건 = Document 1건, `metadata`에 표 ID 보존 |
| Chroma / FAISS | BM25 (`BM25TableRetriever`, 오프라인). 임베딩·Chroma는 T2-05에서 같은 자리에 꽂는다 |
| ChatOpenAI | 실전 1 `LLMClient` (HCX / mock)를 Runnable로 감쌈 |
| query_rag | `kosis_rag.chain.answer` + LangChain tool (`search_tables`, `answer_table_question`) |
| Gradio | 제외 — 프론트는 Next.js (T2-11) |

In [ ]:
# Make the repo modules importable (same paths as pytest.ini).
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
for sub in (".", "src", "vendor/phase1", "vendor/phase1/src"):
    if str(ROOT / sub) not in sys.path:
        sys.path.insert(0, str(ROOT / sub))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)

# data/ is gitignored; point KOSIS_DATA_DIR at the Phase 1 data folder if it lives elsewhere.
DATA_DIR = Path(os.environ.get("KOSIS_DATA_DIR", ROOT / "data"))
CATALOG = DATA_DIR / "tables.csv"
if not CATALOG.exists():
    CATALOG = ROOT / "tests" / "fixtures" / "tables_sample.csv"  # 4-row sample so the notebook still runs
print(CATALOG)

## 1. 데이터 로드와 점검 (실전 1 출력 형식 그대로)

In [ ]:
from kosis_rag.loader import load_catalog

tables = load_catalog(CATALOG)  # every row validated by TableInfo; a bad row raises here
print("통계표 수:", len(tables))
tables[0]

In [ ]:
import pandas as pd

df = pd.DataFrame([t.model_dump() for t in tables])
print(df[["survey_name", "organization"]].value_counts().head(10))
df["coverage"].value_counts()

## 2. 문서화 — 표 1건 = Document 1건

In [ ]:
from kosis_rag.loader import to_document

docs = [to_document(t) for t in tables]
lengths = [len(d.page_content) for d in docs]
print("문서 수:", len(docs), "| 길이 최소/평균/최대:", min(lengths), sum(lengths) // len(lengths), max(lengths))
print(docs[0].page_content)
docs[0].metadata

## 3. 색인과 검색기

In [ ]:
from kosis_rag.retriever import BM25TableRetriever, tokenize

retriever = BM25TableRetriever.from_documents(docs)
print(tokenize("최근 3개월 출생아수"))
for d in retriever.invoke("시군구별 주민등록세대수")[:3]:
    print(round(d.metadata["score"], 2), d.metadata["tbl_id"], d.metadata["table_name"])

## 4. LLM과 RAG 체인

`KOSIS_LLM_BACKEND=hcx`면 HCX(`HCX_API_KEY` 필요), 기본은 mock. 프롬프트는 `kosis_rag.chain.PROMPT`.

In [ ]:
from kosis_chat.config import load_settings
from kosis_chat.llm.model import build_client
from kosis_rag import chain

settings = load_settings(backend=os.environ.get("KOSIS_LLM_BACKEND", "mock"))
client = build_client(settings)
print(client.name)
print(chain.PROMPT.format(question="<질문>", context="<후보 통계표>"))

In [ ]:
result = chain.answer(retriever, client, "시군구별 주민등록세대수를 보고 싶어", k=5)
print(result.answer_origin, result.llm_status)
print(result.answer)

## 5. LangChain tool — 관측·평가의 단위

T2-03 하네스와 종합 단계 에이전트는 이 tool만 호출한다. 콜백(LangSmith, OpenTelemetry)을 `config={"callbacks": [...]}`로 넘기면
tool → retriever → chain → `phase1_llm` 순서의 run 트리가 남는다.

In [ ]:
from kosis_rag.tools import build_tools

search_tables, answer_table_question = build_tools(retriever, client)
print(search_tables.name, search_tables.args)
hits = search_tables.invoke({"question": "주민등록 총인구수", "k": 5})
pd.DataFrame(hits)[["tbl_id", "table_name", "period", "score"]]

In [ ]:
# What the harness will grade: candidate table ids in rank order.
[h["tbl_id"] for h in hits]

## 6. 테스트 — 답 있음 / 답 없음 / 역할 밖

In [ ]:
for q in ["시군구별 주민등록세대수", "23년 9월 서울 집값 추이", "bitcoin price", "너는 뭘 하는 챗봇이니?"]:
    r = answer_table_question.invoke({"question": q, "k": 3})
    print(f"Q: {q}\n  no_answer={r['no_answer']} origin={r['answer_origin']} 후보={[c['tbl_id'] for c in r['candidates']]}\n")

## 다음 태스크가 바꿀 자리

- T2-04 `tokenize` → 한국어 토크나이저 비교 · T2-05 `BM25TableRetriever` 자리에 Chroma 검색기
- T2-06 하이브리드(RRF) · T2-07 리랭커: `retriever` 인자만 교체
- T2-09·T2-14 답 없음 임계값, 기간·주기 불일치 거절: `chain.answer`의 TODO

## ChromaDB 검색 → 답 (T2-26)

T2-24 컬렉션(KURE-v1로 임베딩)에서 표를 찾고 같은 체인으로 답한다. `docker compose up -d chroma`와 `.env`의 `CHROMA_COLLECTION`이 필요하다.
임베딩 모델·토크나이저는 임시값이다 — `kosis_rag/index.py`의 `EMBED_MODEL` TODO(T2-05) 참고.

In [ ]:
from kosis_rag.retriever import ChromaTableRetriever

chroma = ChromaTableRetriever.connect()  # fails fast if the server is down or the collection was embedded with another model
for d in chroma.invoke("시군구별 주민등록세대수")[:3]:
    print(round(d.metadata["score"], 3), d.metadata["tbl_id"], d.metadata["table_name"])

result = chain.answer(chroma, client, "시군구별 주민등록세대수를 보고 싶어", k=5)
print(result.answer_origin, result.llm_status)
print(result.answer)